<!--
Copyright (c) 2020-2024 Key4hep-Project.

Licensed under the Apache License, Version 2.0.
-->

# Hands-on 6: reading EDM4hep hits and contributions

This exercise reads the **simplecalo2** EDM4hep file in Python and builds the total energy spectrum, longitudinal shower profile, lateral shower shape, and contribution timing distribution.

There are six questions, marked `Q1` to `Q6`. The completed notebook is `readEdm4hepSolution.ipynb`. Uproot reads the events, DD4hep decodes cell IDs, and Awkward Array, NumPy, and Matplotlib handle the analysis and plots.

In [ ]:
import awkward as ak
import dd4hep
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import uproot

from drdcalo_tutorials import simplecalo2_input

## Open the event data

The generated 500-event file is used when available; otherwise the notebook uses the bundled 10-event sample. Set `SIMPLECALO2_FILE` to choose another compatible file.

In [ ]:
input_file = simplecalo2_input()

events = uproot.open(input_file, handler=uproot.MultithreadedFileSource)["events"]
print(f"Reading {events.num_entries} events from {input_file}")

## Q1 — decode the cell ID

Each hit carries a 64-bit `cellID`. Which encoding string belongs to `simplecaloRO`? Find the `<id>` element in the `<readout>` block of `simplecalo2/compact/simplecalo2.xml`, then pass it to DD4hep's native bit-field decoder.

In [ ]:
# Q1
ENCODING = "FILL ME"
decoder = dd4hep.core.DDSegmentation.BitFieldCoder(ENCODING)

## Q2 and Q3 — read and decode the hits

Uproot exposes each EDM4hep member as an Awkward array with one nested list per event. For Q2, replace `FILL ME` with the readout collection name. For Q3, use the `decode` helper below to extract `calolayer`, `abslayer`, and `cellid` from the flattened IDs.

In [ ]:
# Q2
COLLECTION = "FILL ME"

def read(field, collection=COLLECTION):
    return events[f"{collection}/{collection}.{field}"].array(library="ak")

cell_ids = read("cellID")
hit_energies = read("energy")
hit_x = read("position.x")
hit_y = read("position.y")

# Q3
# The decoder knows where each field sits inside the 64-bit cellID. Applying that
# shift and mask with NumPy decodes the whole array at once, which is what keeps
# this cell fast once you point the notebook at a full 500-event simulation.
def decode(field, ids):
    element = decoder[field]
    values = (ids >> element.offset()) & ((1 << element.width()) - 1)
    if element.isSigned():  # simplecalo1 encodes x and y as signed fields
        sign_bit = 1 << (element.width() - 1)
        values = (values ^ sign_bit) - sign_bit
    return values

flat_cell_ids = ak.to_numpy(ak.flatten(cell_ids))
calo_layer = ...  # FILL ME: decode("...", flat_cell_ids)
abs_layer = ...   # FILL ME
sub_cell_id = ... # FILL ME

N_LAYERS = 20  # LayersNumber in simplecalo2.xml
N_CELLS = 10   # SensLayerX / CellX
cell_x = sub_cell_id % N_CELLS
cell_y = sub_cell_id // N_CELLS

## Q4 — total energy and longitudinal profile

Use `ak.sum(..., axis=1)` to keep one total per event. Flatten the hit energies with `ak.flatten`, then use the decoded layer array and `np.histogram(..., weights=...)` for the layer profile. Divide the profile by the number of events.

In [ ]:
# Q4
n_events = events.num_entries
total_energies = ...  # FILL ME
energy = ...         # FILL ME
layer = calo_layer

energy_max = 1.2 * total_energies.max() if total_energies.max() > 0 else 1.0
energy_counts, energy_edges = np.histogram(total_energies, bins=100, range=(0, energy_max))
layer_edges = np.arange(0.5, N_LAYERS + 1.5)
average_layer_energy, _ = ...  # FILL ME
average_layer_energy /= n_events

print(f"Mean total energy: {total_energies.mean():.3f} GeV over {n_events} events")

## Q5 — lateral shower shape

The cell placement uses the outer loop for *y* and the inner loop for *x*, so `cellid = 10 × iY + iX`. Build one weighted `np.histogram2d` per layer and divide it by the number of events.

In [ ]:
# Q5
ix = cell_x
iy = cell_y
lateral_energy = []

for layer_number in range(1, N_LAYERS + 1):
    in_layer = layer == layer_number
    histogram, _, _ = ...  # FILL ME
    lateral_energy.append(histogram / n_events)

lateral_energy = np.stack(lateral_energy)

## Q6 — contribution timing

EDM4hep stores the hit contributions in the companion collection `<hit collection>Contributions`. Read its `time` and `energy` fields, flatten them, and make a 50-bin weighted histogram from 0 to 10 ns. Each contribution also carries `stepPosition.x/y/z` and `stepLength`, the position and length of the Geant4 step that produced it, in mm.

In [ ]:
# Q6
CONTRIBUTIONS = f"{COLLECTION}Contributions"
contribution_times = ...     # FILL ME
contribution_energies = ...  # FILL ME
time_energy, time_edges = ...  # FILL ME
print(f"Read {len(contribution_times)} contributions")

## Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].stairs(energy_counts, energy_edges, fill=True)
axes[0].set(xlabel="Energy [GeV]", ylabel="Events", title="Total energy deposit")
axes[1].bar(range(1, N_LAYERS + 1), average_layer_energy)
axes[1].set(xlabel="Layer", ylabel="Energy [GeV]", title="Average energy deposit per layer")
fig.tight_layout()

In [ ]:
busiest = int(np.argmax(lateral_energy.sum(axis=(1, 2))))
lateral = lateral_energy[busiest]
print(f"Most energetic layer: {busiest + 1}")
print(f"Hottest cell: {100 * lateral.max() / lateral.sum():.1f}% of that layer's energy")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# Brass has a Moliere radius of about 2 cm against 10 cm cells, so the impact
# cell takes almost everything. A log scale is what makes the tails visible.
image = axes[0].imshow(
    np.ma.masked_less_equal(lateral.T, 0), origin="lower",
    extent=(0, N_CELLS, 0, N_CELLS), aspect="equal",
    norm=LogNorm(vmin=lateral.max() * 1e-5, vmax=lateral.max()),
)
axes[0].set(xlabel=r"$i_X$", ylabel=r"$i_Y$", title=f"Lateral shape, layer {busiest + 1}")
fig.colorbar(image, ax=axes[0], label="Average energy [GeV]")
axes[1].stairs(time_energy, time_edges, fill=True)
axes[1].set(xlabel="Time [ns]", ylabel="Energy [GeV]", title="Hit contribution timing")
fig.tight_layout()

## Cross-check the cell indices

The hottest hit in the first event should have the same stored position as the cell centre implied by its decoded indices.

In [ ]:
hottest = int(np.argmax(ak.to_numpy(hit_energies[0])))
sub = decoder.get(int(cell_ids[0][hottest]), "cellid")
i_x, i_y = sub % N_CELLS, sub // N_CELLS

# EDM4hep stores positions in mm (see the EDM4hep yaml file), so these are the
# XML dimensions CellX and SensLayerX / 2 expressed in mm.
CELL = 100.0  # mm
HALF = 500.0  # mm
implied_x = -HALF + CELL / 2 + i_x * CELL
implied_y = HALF - CELL / 2 - i_y * CELL
stored_x, stored_y = float(hit_x[0][hottest]), float(hit_y[0][hottest])

print(f"hottest cell: cellid={sub} -> iX={i_x}, iY={i_y}")
print(f"  implied centre : x={implied_x:+7.1f} mm, y={implied_y:+7.1f} mm")
print(f"  stored position: x={stored_x:+7.1f} mm, y={stored_y:+7.1f} mm")

if np.isclose(implied_x, stored_x) and np.isclose(implied_y, stored_y):
    print("  -> match: the cell indices are decoded correctly")
else:
    print("  -> MISMATCH: this input is not a simplecalo2 file carrying the")
    print("     cells placed in Hands-on 4. Check which file was opened above.")